# edit 赛道 · 数据审阅（审计复核案例 + hard++++ 批次验收）

消费 `audit_doublecheck.py` 三件产物（sample / results / report），逐案例对照 **原图 vs 审计自报 vs 复核修正**，用于：
- 人工确认复核判读是否成立（防复核自身误杀）；
- 为出题驱动的素材注入策略提供直觉（哪些维度虚、虚成什么样）。

数据源（只读）：`data/audit_doublecheck_sample.jsonl`（40 张分层抽样，含审计原值）、`data/audit_doublecheck_results.jsonl`（复核修正）、`data/audit_doublecheck.report.json`（汇总结论）、`data/focus200/manifest.jsonl`（sha256→图路径权威映射）。

复核头条：**不可信率 40%（16/40）> 20% 阈值，审计整体偏乐观成立**；scene 修正比 ~0.95；膨胀重灾 = 环境作用/过程时刻；载体条目误报率 11.5% < 15%（载体清单不需重算）。

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

EDIT_DIR = Path('/tank/demiwtg/benchmark/edit')
DATA = EDIT_DIR / 'data'

report = json.loads((DATA / 'audit_doublecheck.report.json').read_text())
sample = {json.loads(l)['audit']['sha256']: json.loads(l) for l in (DATA / 'audit_doublecheck_sample.jsonl').open()}
results = {json.loads(l)['sha256']: json.loads(l) for l in (DATA / 'audit_doublecheck_results.jsonl').open()}
manifest = {json.loads(l)['sha256']: json.loads(l) for l in (DATA / 'focus200' / 'manifest.jsonl').open()}

print(f"样本 {report['sample_size']} 张 · 不可信 {report['untrusted_count']}（{report['untrusted_rate']:.0%}） · 判读规则 {report['decision_rule']}")
print('整体判读分布:', report['overall_counts'])
print('scene 计数: 原始均值 %.2f → 修正 %.2f（比例 %.3f；强项 %.3f）' % (
    report['scene_counts']['original_mean'], report['scene_counts']['corrected_mean'],
    report['scene_counts']['corrected_to_original_ratio'], report['scene_counts']['corrected_strong_to_original_ratio']))
print('膨胀维度 Top:', dict(sorted(report['scene_inflation_dims'].items(), key=lambda x: -x[1])[:5]))
print('载体: 条目误报率 %.1f%%（%d/%d） · 行误报率 %.1f%%（仅诊断） · 判定 %s' % (
    report['carriers']['item_false_positive_rate']*100, report['carriers']['wrong_items'], report['carriers']['claimed_items'],
    report['carriers']['row_false_positive_rate']*100, report['decisions']))

样本 40 张 · 不可信 16（40%） · 判读规则 scene_inflation>=3 or groups_wrong>=2
整体判读分布: {'不可信': 16, '偏乐观': 24}
scene 计数: 原始均值 9.12 → 修正 8.70（比例 0.953；强项 0.930）
膨胀维度 Top: {'环境作用': 19, '过程时刻': 9, '动态要素': 2, '纵深层次': 2, '视点剖示': 2}
载体: 条目误报率 11.5%（7/61） · 行误报率 17.5%（仅诊断） · 判定 {'audit_is_optimistic': True, 'carrier_supply_requires_reestimate_by_item_rate': False, 'carrier_rows_need_review_by_row_rate': True}


## 案例审阅（图 + 审计自报 vs 复核修正）

红色 = 审计虚报（复核判错/判降）；蓝色 = 审计漏报（复核补上，方向相反）。

In [2]:
def _li(items, color, tag):
    return ''.join(f"<li style='color:{color}'><b>[{tag}]</b> {x if isinstance(x,str) else x.get('class','')+'：'+x.get('问题','')}</li>" for x in items)

def display_case(sample_id):
    s = next(v for v in sample.values() if v['sample_id'] == sample_id)
    sha = s['audit']['sha256']; r = results[sha]; a = s['audit']
    img = (DATA / 'focus200' / manifest[sha]['file']) if sha in manifest else Path(a['path'])
    badge = '#c0392b' if r['overall'] == '不可信' else '#b8860b'
    groups = ''.join(f"<tr><td>{g['class']}</td><td>{g['count']}</td><td>{g['role']}</td><td>{g['variants']}/{g['arrangement']}</td></tr>" for g in a['same_class_groups'])
    html = f"""
    <div style='border:1px solid #ccc; padding:10px; margin:6px 0; font-size:13px'>
      <div style='font-size:15px'><b>{sample_id} · {a['instance']}</b>
        <span style='background:{badge}; color:#fff; padding:1px 8px; border-radius:3px'>{r['overall']}</span>
        <span style='color:#888'> · 分层 {s['primary_stratum']} · scene {a['scene_count']}（强 {a['scene_strong']}） · 摘要：{a['summary']}</span></div>
      <div style='display:flex; gap:14px; margin-top:8px'>
        <div style='flex:0 0 460px'><img src='{img}' style='max-width:460px; max-height:460px'></div>
        <div style='flex:1'>
          <b>审计自报同类组</b>
          <table border='1' cellpadding='3' style='border-collapse:collapse; font-size:12px'>
            <tr><th>class</th><th>count</th><th>role</th><th>variants/arr</th></tr>{groups}</table>
          <div style='margin-top:4px'><b>参照物</b>: {', '.join(a['referents']) or '—'} · <b>载体</b>: {', '.join(a['consequence_carriers']) or '—'}</div>
          <hr style='margin:6px 0'>
          <b>复核修正</b>
          <ul style='margin:2px 0; padding-left:18px'>
            {_li(r['scene_inflation'], '#c0392b', 'scene虚')}
            {_li(r['groups_wrong'], '#c0392b', '组错')}
            {_li(r['carriers_wrong'], '#c0392b', '载体虚')}
            {_li(r['suitability_wrong'], '#c0392b', '适配虚')}
            {_li(r['scene_missed'], '#2471a3', 'scene漏')}
            {_li(r['groups_missed'], '#2471a3', '组漏')}
            {_li(r['carriers_missed'], '#2471a3', '载体漏')}
          </ul>
          <i style='color:#555'>备注：{r.get('notes','')}</i>
        </div>
      </div>
    </div>"""
    display(HTML(html))

print('display_case(样本ID) 就绪，可用样本 ID：', sorted(v['sample_id'] for v in sample.values()))

display_case(样本ID) 就绪，可用样本 ID： ['DC01', 'DC02', 'DC03', 'DC04', 'DC05', 'DC06', 'DC07', 'DC08', 'DC09', 'DC10', 'DC11', 'DC12', 'DC13', 'DC14', 'DC15', 'DC16', 'DC17', 'DC18', 'DC19', 'DC20', 'DC21', 'DC22', 'DC23', 'DC24', 'DC25', 'DC26', 'DC27', 'DC28', 'DC29', 'DC30', 'DC31', 'DC32', 'DC33', 'DC34', 'DC35', 'DC36', 'DC37', 'DC38', 'DC39', 'DC40']


In [3]:
CURATED = ['DC01', 'DC05', 'DC12', 'DC20', 'DC36', 'DC09']   # ← 改这里：要看的样本 ID（覆盖全部失败通道）

for sid in CURATED:
    display_case(sid)

class,count,role,variants/arr
黄色工程车,5,is_subject,identical/row
橙色工装工人,2,is_subject,varied/cluster
斜拉索,10,unrelated,identical/row


class,count,role,variants/arr
高尔夫球包,2,unrelated,varied/cluster
观众,8,unrelated,varied/row


class,count,role,variants/arr


class,count,role,variants/arr


class,count,role,variants/arr
科研人员,3,is_subject,varied/cluster
军绿色制服,3,unrelated,identical/cluster
帽子,3,unrelated,varied/cluster
背包,3,unrelated,varied/cluster


class,count,role,variants/arr
自行车手,3,is_subject,varied/row
自行车,3,is_subject,varied/row
观众,100,unrelated,varied/cluster
旗帜,10,unrelated,varied/scattered


## 全量透视（40 张内失败模式分布）

In [4]:
rows = []
for sha, r in results.items():
    s = sample[sha]
    for g in r['groups_wrong']:
        kind = 'role' if 'role' in g['问题'] else ('count不可核' if '不可核' in g['问题'] else ('count多计' if '多计' in g['问题'] else 'count少计'))
        rows.append({'sha': sha, 'instance': r['instance'], 'type': '组.' + kind})
    rows.append({'sha': sha, 'instance': r['instance'], 'type': '载体虚报'}) if r['carriers_wrong'] else None
gdf = pd.DataFrame(rows)

print('== 同类组错误类型分布 =='); print(gdf[gdf.type.str.startswith('组')].type.value_counts().to_string())
print('\n== scene 膨胀维度分布 =='); print(pd.Series(report['scene_inflation_dims']).sort_values(ascending=False).to_string())
print('\n== 分层不可信率 ==')
print(pd.DataFrame([{'分层': k, 'n': v['n'], '不可信': v['untrusted'], '率': f"{v['untrusted_rate']:.0%}"}
                    for k, v in report['by_primary_stratum'].items()]).to_string(index=False))
print('\n== 按 generator ==')
print(pd.DataFrame([{'generator': k, 'n': v['n'], '不可信率': f"{v['untrusted_rate']:.0%}", '载体虚': v['carrier_wrong_items']}
                    for k, v in report['by_generator'].items()]).to_string(index=False))

== 同类组错误类型分布 ==
type
组.count少计     34
组.role        13
组.count不可核     1
组.count多计      1

== scene 膨胀维度分布 ==
环境作用     19
过程时刻      9
动态要素      2
纵深层次      2
视点剖示      2
多人物编排     1
细节密度      1
交互链       1
多实例对比     1

== 分层不可信率 ==
               分层  n  不可信   率
 gpt_carriers_ge2  7    2 29%
      scene_11_12 10    7 70%
        scene_6_7 10    3 30%
        scene_8_9 10    2 20%
subject_group_ge3  3    2 67%

== 按 generator ==
  generator  n 不可信率  载体虚
 qwen-image 32  41%    4
gpt-image-2  8  38%    3


## 对出题驱动的含义（备忘）

1. **素材清单以候选注入**：审计字段标注“含虚项，以图为准，未核实不得引用”，防同源锚定传染（审计与出题均 Qwen 系）。
2. **环境作用/过程时刻降权**：膨胀重灾维度；L3 难度背书若锚在这两维须对图坐实。交叉核验容差按 0.95 放宽，防误杀好题。
3. **载体 11.5% 虚报（镜面反光/倒影/影子类为高发）**：L3 传播链的载体必须对图确认后才能进后果链。
4. count 类错误以**少计**为主（复核偏保守方向），对出题影响弱于虚报；但“人群不可核”提示大 count 组不应作为定位锚。

## hard++++ 补生成批次审阅（quality_regen_v1）

codex 用 image_gen 合成的高难度源图池：100 计划（与双载体补产 93 实例不相交）/ **89 过 / 11 拒（89%）**；
验收硬门：主体可辨完整可分离、3-5 个可数同类变体、恰两个参照物、**≥2 类载体须图面实证**（影子要映像、液体要液面、机构要关节）、6-9 个复杂度维（防双计）、零可读文字/logo。
注意：**该批次尚未晋升进 focus200 manifest**（单进程审后晋升）；89 张是 hard++++ 出题的唯一合格池。

In [5]:
QRV = DATA / 'focus200' / 'quality_regen_v1'
qrv_rows = {json.loads(l)['candidate_id']: json.loads(l) for l in (QRV / 'merged_results.jsonl').open()}
qrv_rev = {}
for _sh in 'abc':
    qrv_rev.update({json.loads(l)['candidate_id']: json.loads(l) for l in (QRV / f'agent_{_sh}/review_r2.jsonl').open()})
qrv_acc_ids = {json.loads(l)['candidate_id'] for l in (QRV / 'accepted_results.jsonl').open()}
qrv_sum = json.loads((QRV / 'review_summary.json').read_text())

print(f"计划 {qrv_sum['planned']} · 过 {qrv_sum['accepted']} · 拒 {qrv_sum['rejected']}（{qrv_sum['acceptance_rate']:.0%}） · 分片", {k: f"{v['accepted']}/{v['planned']}" for k, v in qrv_sum['by_shard'].items()})
print('拒绝原因分布:', qrv_sum['rejection_reason_counts'])

计划 100 · 过 89 · 拒 11（89%） · 分片 {'a': '31/34', 'b': '27/33', 'c': '31/33'}
拒绝原因分布: {'CARRIER_COUNT_LT_2': 3, 'CARRIER_EVIDENCE_UNREADABLE': 1, 'CARRIER_FALSE_POSITIVE': 2, 'FORBIDDEN_TEXT_OR_MARK': 2, 'INSTANCE_AMBIGUOUS_OR_ABSTRACT': 4, 'PERSON_COUNT_MISMATCH': 1, 'REFERENT_AMBIGUOUS': 2, 'SUBJECT_CROPPED': 1, 'TARGET_GROUP_COUNT_MISMATCH': 1}


In [6]:
def display_qrv1(cid):
    row = qrv_rows[cid]; rev = qrv_rev.get(cid, {})
    img = QRV / row['file']
    ok = cid in qrv_acc_ids
    badge = '#1e8449' if ok else '#c0392b'
    reasons = ''.join(f"<li style='color:#c0392b'><b>拒</b> {x}</li>" for x in rev.get('reject_reasons', []))
    html = f"""
    <div style='border:1px solid #ccc; padding:10px; margin:6px 0; font-size:13px'>
      <div style='font-size:15px'><b>{cid} · {row['instance']}</b>
        <span style='background:{badge}; color:#fff; padding:1px 8px; border-radius:3px'>{'过' if ok else '拒'}</span>
        <span style='color:#888'> · shard {row['shard']} · {row['priority']} · 观测同类数 {rev.get('observed_count','—')} · 载体核实 {', '.join(rev.get('carriers_verified', [])) or '—'}</span></div>
      <div style='display:flex; gap:14px; margin-top:8px'>
        <div style='flex:0 0 460px'>{f"<img src='{img}' style='max-width:460px; max-height:460px'>" if img.exists() else '<i>图缺失</i>'}</div>
        <div style='flex:1'>
          <div><b>要求载体</b>: {', '.join(row['carriers_required'])} · <b>参照物</b>: {', '.join(row['acceptance']['unique_referents'])} · <b>复杂度目标</b>: {row['acceptance']['complexity_dimensions']['target']} 维</div>
          <hr style='margin:6px 0'>
          <ul style='margin:2px 0; padding-left:18px'>{reasons}</ul>
          <i style='color:#555'>复核备注：{rev.get('notes','')}</i>
        </div>
      </div>
    </div>"""
    display(HTML(html))

print('display_qrv1(candidate_id) 就绪')

display_qrv1(candidate_id) 就绪


In [7]:
import random as _r
SHOW_ACCEPTED = 8    # ← 改这里：随机抽几张过审图（random_state=42）
SHOW_REJECTED = True # ← 改这里：是否附全部 11 张拒收图（建议亲验拒得对不对）

_acc = sorted(qrv_acc_ids); _r.Random(42).shuffle(_acc)
for _cid in _acc[:SHOW_ACCEPTED]:
    display_qrv1(_cid)
if SHOW_REJECTED:
    display(HTML('<h3>拒收案例（11 张）</h3>'))
    for _cid in sorted(c for c, r in qrv_rev.items() if not r['accepted']):
        display_qrv1(_cid)